Use the SimLM model to help create a score based on the relevant data presented by the Ardook

Model Link: https://huggingface.co/intfloat/simlm-msmarco-reranker

In [4]:
#Imports and Installation
!pip install transformers

import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BatchEncoding, PreTrainedTokenizerFast
from transformers.modeling_outputs import SequenceClassifierOutput
import torch
import pandas as pd
import numpy as np


In [5]:
def encode(tokenizer: PreTrainedTokenizerFast,
           query: str, passage: str, title: str = '-') -> BatchEncoding:
    return tokenizer(query,
                     text_pair='{}: {}'.format(title, passage),
                     max_length=192,
                     padding=True,
                     truncation=True,
                     return_tensors='pt')

tokenizer = AutoTokenizer.from_pretrained('intfloat/simlm-msmarco-reranker')
model = AutoModelForSequenceClassification.from_pretrained('intfloat/simlm-msmarco-reranker')
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [6]:
FILE_PATH  = "/content/Copy of Ardook_Data_All_Grant_Recommendations_UPDATED_v3 (2).xlsx"   # path after uploading data
SHEET_NAME = "Grant Tracker"
TEXT_COL   = "Core Requirements"

In [7]:
df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
df.columns = df.columns.str.strip()
print(df.shape)
print("Empty requirement cells:", df[TEXT_COL].isna().sum())

# Creates the table
data = df[["Priority / Deadline Order", "Program", TEXT_COL, "Fit"]].copy()
data[TEXT_COL] = data[TEXT_COL].fillna("").astype(str).str.strip()
data.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/Copy of Ardook_Data_All_Grant_Recommendations_UPDATED_v3 (2).xlsx'

In [ ]:
#Input Comparisons
with torch.no_grad():
    bat_di = encode(tokenizer, 'how many movies can we watch over the weekend?', 'Each movie is 2 hours long and the weekend is 48 hours total so we can watch 24 movies.')
    outputs: SequenceClassifierOutput = model(**bat_di, return_dict=True)
    print(outputs.logits[0])

    bat_di = encode(tokenizer, 'how many movies can we watch over the weekend?', 'The theater is about 12 hours away and popcorn costs $5.')
    outputs: SequenceClassifierOutput = model(**bat_di, return_dict=True)
    print(outputs.logits[0])